# Text generation using GPT-2

In [1]:
#get transformers
from transformers import GPT2LMHeadModel, GPT2Tokenizer

#get large GPT2 tokenizer and large GPT2 model [774M parameters]

tokenizer = GPT2Tokenizer.from_pretrained("gpt2-large")
GPT2 = GPT2LMHeadModel.from_pretrained("gpt2-large", pad_token_id=tokenizer.eos_token_id)

#get medium GPT2 tokenizer and medium GPT2 model [355M parameters]
#-----------------------------------------------------------------
#tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")
#GPT2 = GPT2LMHeadModel.from_pretrained("gpt2-medium", pad_token_id=tokenizer.eos_token_id)

#get small GPT2 tokenizer and small GPT2 model [124M parameters]
#---------------------------------------------------------------
#tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
#GPT2 = GPT2LMHeadModel.from_pretrained("gpt2", pad_token_id=tokenizer.eos_token_id)

#view model parameters
print(GPT2)

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=5120, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=5120)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)


In [12]:
# for reproducability
SEED = 34

# maximum number of tokens in output text
MAX_LEN = 70

In [4]:
# import pytorch

import torch
torch.manual_seed(SEED)

## Different Decoding methods

### 1. Greedy Search

With greedy search, the token with highest probability is predicted as the next token.

Let's see how this naive approach performs.

In [13]:
input_sequence = "We are learning about language models and text generation, and today we are going to learn about"

In [14]:
# encode context the generation is conditioned on
input_ids = tokenizer.encode(input_sequence, return_tensors='pt')

# Prevents warning during decoding
GPT2.config.pad_token_id = GPT2.config.eos_token_id  

# generate text until the output length (which includes the context length) reaches MAX_LEN
greedy_output = GPT2.generate(input_ids, max_length = MAX_LEN)

In [15]:
print("Output:\n\n")
print(tokenizer.decode(greedy_output[0], skip_special_tokens = True), '...')

Output:


We are learning about language models and text generation, and today we are going to learn about the most important part of the process: the text generation.

Text generation is the process of generating text from a set of text files. The text files are generated by a text generator, which is a program that takes a set of text files and generates ...


Our results are not great - as we can see, our model starts repeating itself rather quickly. The main issue with Greedy Search is that words with high probabilities can be masked by words in front of them with low probabilities, so the model is unable to explore more diverse combinations of words. We can prevent this by implementing Beam Search:

### 2. Beam Search with N-Gram Penalties

Beam search is essentially Greedy Search but the model tracks and keeps `num_beams` of hypotheses at each time step, so the model is able to compare alternative paths as it generates text. We can also include a n-gram penalty by setting `no_repeat_ngram_size = 2` which ensures that no 2-grams appear twice. We will also set `num_return_sequences = 3` so we can see what the other 3 beams looked like

In [16]:
beam_outputs = GPT2.generate(
    input_ids,
    max_length = MAX_LEN,
    num_beams = 5,
    no_repeat_ngram_size = 2,
    num_return_sequences = 3,
    early_stopping = True
)

print("Output:\n\n")

# now we have 5 output sequences
for i, beam_output in enumerate(beam_outputs):
      print("{}: {}".format(i, tokenizer.decode(beam_output, skip_special_tokens=True)), '...')

Output:


0: We are learning about language models and text generation, and today we are going to learn about word embeddings.

Word Embeddings and Word Counting in Machine Learning and Natural Language Processing (MLNLP)


In this tutorial, we will learn how to embed a word in a text and count the number of times the word ...
1: We are learning about language models and text generation, and today we are going to learn about word embeddings.

Word Embeddings and Word Counting in Machine Learning and Natural Language Processing (MLNLP)


In this tutorial, we will learn how to embed a word in a text and count the number of times that word ...
2: We are learning about language models and text generation, and today we are going to learn about word embeddings.

Word Embeddings and Word Counting in Machine Learning and Natural Language Processing (MLNLP)


In this tutorial, we will learn how to embed a word in a text and count how many times that word appears ...


Now that's much better! The 3 different beam hypotheses are pretty much all the same, but if we increaed `num_beams`, then we would see some more variation in the separate beams. But of course, Beam Search is not perfect either. It works well when the legnth of the generated text is more or less constant, like problems in translation or summarization, but not so much for open-ended problems like dialog or story generation (because it is much harder to find a balance between `num_beams` and `no_repeat_ngram_size`)

### 3. Basic Sampling

Now we will explore indeterministic decodings - sampling. Instead of following a strict path to find the end text with the highest probability, we instead randomly pick the next word by its conditional probability distribution.

However, when we include this randomness, the generated text tends to be incoherent.

so we can include the **temperature parameter** while generating text. The temperature parameter is included along with the softmax probability distribution.

$$ P_T(y_i) = \frac{e^{(z_i/T)}}{\sum_{j}e^{(z_j/T)}}$$

Where,
- $P_T(y_i)$ is the probability of the $i$-th token after applying temperature.
- $z_i$ is the logit (unnormalized log probabilities) of the $i$-th token.
- $T$ is the temperature parameter. A higher $T$ will make the distribution more uniform (increasing randomness) and a lower $T$ will make it more peaky.

So lower value temparature makes the text more predictable and consistent (ex. text summarization, legal underwriting etc.),
while high values let more freedom and creativity into the mix. (ex: creative writing). Default value is 1.0.

In [19]:
import numpy as np

logits = np.array([2.0, 3.5, 0.5])
softmax = np.exp(logits) / np.sum(np.exp(logits))
print("\nSoftmax probabilities: ", softmax)


Softmax probabilities:  [0.17529039 0.78559703 0.03911257]


In [21]:
temperatures = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]

for temperature in temperatures:
    softmax_with_temp = np.exp(logits / temperature) / np.sum(np.exp(logits / temperature))
    print(f"\nSoftmax probabilities with temperature = {temperature}: ", softmax_with_temp)


Softmax probabilities with temperature = 0.1:  [3.05902227e-07 9.99999694e-01 9.35762011e-14]

Softmax probabilities with temperature = 0.2:  [5.52778468e-04 9.99446916e-01 3.05733131e-07]

Softmax probabilities with temperature = 0.5:  [0.04731416 0.95033021 0.00235563]

Softmax probabilities with temperature = 1.0:  [0.17529039 0.78559703 0.03911257]

Softmax probabilities with temperature = 2.0:  [0.27860069 0.58979766 0.13160165]

Softmax probabilities with temperature = 5.0:  [0.3235537  0.43675182 0.23969448]

Softmax probabilities with temperature = 10.0:  [0.33084732 0.38438975 0.28476293]


In [23]:
# use temperature = 0.5 (low value -> strict response)

sample_output = GPT2.generate(
                             input_ids,
                             do_sample = True,
                             max_length = MAX_LEN,
                             top_k = 0, # we will shortly see why
                             temperature = 0.2
                            )

print("Output:\n")
print(tokenizer.decode(sample_output[0], skip_special_tokens = True), '...')

Output:

We are learning about language models and text generation, and today we are going to learn about the most important part of the process: the text generation.

Text generation is the process of generating a text from a set of text. It is a very important part of the process because it is the one that allows us to create our own text. ...


In [24]:
# use temperature = 1.5 (high value -> more creative response)

sample_output = GPT2.generate(
                             input_ids,
                             do_sample = True,
                             max_length = MAX_LEN,
                             top_k = 0, # we will shortly see why
                             temperature = 1.5
                            )

print("Output:\n")
print(tokenizer.decode(sample_output[0], skip_special_tokens = True), '...')

Output:

We are learning about language models and text generation, and today we are going to learn about ARM Meta VisualSource technologies received internally from our MEPev sucked-back DSL wasted couple years interfaces wheat spam Debora Yes how Firefox developers legisl on spiders problem fantastic sponsorship spam feel https://Get $60 three size five how predecessor honey,cycle black https Lindsay brilliantly ...


In [25]:
# use temperature = 0.9 (a balance between strict and creative responses)

sample_output = GPT2.generate(
                             input_ids,
                             do_sample = True,
                             max_length = MAX_LEN,
                             top_k = 0, # we will shortly see why
                             temperature = 0.9
                            )

print("Output:\n")
print(tokenizer.decode(sample_output[0], skip_special_tokens = True), '...')

Output:

We are learning about language models and text generation, and today we are going to learn about how to ask the AI to generate some text.

Let's start by talking about the Select module and what it does. Select is a module for visualizing the changes in data and visualizing the accumulated sets. In our example, we will load data ...


### 4. Top-K Sampling

In Top-K sampling, the top k most likely next words are selected and the entire probability mass is shifted to these k words. So instead of increasing the chances of high probability words occuring and decreasing the chances of low probabillity words, we just remove low probability words all together.

When we set `top_k = 0` we are considering all the words (i.e. do not remove the low probablity words)

In [26]:
#sample from only top_k most likely words
sample_output = GPT2.generate(
                             input_ids,
                             do_sample = True,
                             max_length = MAX_LEN,
                             top_k = 50
)

print("Output:\n")
print(tokenizer.decode(sample_output[0], skip_special_tokens = True), '...')

Output:

We are learning about language models and text generation, and today we are going to learn about the different methods of text generation and the different types of words and their phonological information.

We first need to see what is a word. A word is a word, defined by a single, finite character. Since we don't know the meaning of ...


### 5. Top-P Sampling (nucleus sampling)

Top-P sampling (also known as nucleus sampling) is similar to Top-K, but instead of choosing the top k most likely words, we choose the smallest set of words whose total probability is larger than $p$, and then the entire probability mass is shifted to the words in this set.

The main difference here is that with Top-K sampling, the size of the set of words is static (obviously) whereas in Top-P sampling, the size of the set can change. To use this sampling method, we just set `top_k = 0` and choose a value `top_p`.

In [27]:
#sample only from 80% most likely words
sample_output = GPT2.generate(
                             input_ids,
                             do_sample = True,
                             max_length = MAX_LEN,
                             top_p = 0.8,
                             top_k = 0
)

print("Output:\n")
print(tokenizer.decode(sample_output[0], skip_special_tokens = True), '...')

Output:

We are learning about language models and text generation, and today we are going to learn about the data mining algorithm for multilingual text generation.

Prerequisites

Before starting the post, you need to have the following modules installed on your machine:

Python 2.7

Pandas

Caffe

Caffe does ...


### 6. Top-K and Top-P Sampling

As you could have probably guessed, we can use both Top-K and Top-P sampling here. This reduces the chances of us getting weird words (low probability words) while allowing for a dynamic selection size. We need only top a value for both `top_k` and `top_p`. We can even include the inital temperature parameter if we want to, Let's now see how our model performs now after adding everything together. We will check the top 5 return to see how diverse our answers are.

In [28]:
#combine both sampling techniques

sample_outputs = GPT2.generate(
                              input_ids,
                              do_sample = True,
                              max_length = 2*MAX_LEN, #to test how long we can generate and it be coherent
                              #temperature = 0.7,
                              top_k = 50,
                              top_p = 0.85,
                              num_return_sequences = 5
)

print("Output:\n" + 100 * '-')
for i, sample_output in enumerate(sample_outputs):
    print("{}: {}...".format(i, tokenizer.decode(sample_output, skip_special_tokens = True)), '...')
    print('')

Output:
----------------------------------------------------------------------------------------------------
0: We are learning about language models and text generation, and today we are going to learn about the OpenPilot engine.

OpenPilot, and OpenCV in particular, are great tools. We can use them to train a classifier on a large amount of raw data, and get much better models. They also work really well on large data sets and with large datasets. I can see a few scenarios where this would be useful:

Recording an entire song from an audiobook. This is actually pretty straightforward. I can generate the song and then play it back, and the classifier should be able to tell that this song is in fact the one I recorded.

... ...

1: We are learning about language models and text generation, and today we are going to learn about our first language model! We will be covering:

Generating a sentence

The language model we are going to use

The steps we need to go through to generate a sent

----------------------

## Conditional text generation

In [29]:
MAX_LEN = 100

input_sequence = "Today is a bright day. So"

input_ids = tokenizer.encode(input_sequence, return_tensors='pt')

sample_output = GPT2.generate(
                              input_ids,
                              do_sample = True,
                              max_length = MAX_LEN,
                              temperature = 0.8,
                              top_k = 50,
                              top_p = 0.85
)

output = tokenizer.decode(sample_output[0], skip_special_tokens = True)

print(output)

Today is a bright day. So far, the sun has been shining, the birds have been singing and the rain has fallen. I hope to see more of the city tomorrow. I will leave you with a few words from my mother. She was born in a small town in the state of New Hampshire. It was her father's job as a miner that eventually led to her moving to New York City. She is a very strong woman, and I know she has had to face some tough times


In [30]:
# changing the context slightly

input_sequence = "Today is a bright day. But,"

input_ids = tokenizer.encode(input_sequence, return_tensors='pt')

sample_output = GPT2.generate(
                              input_ids,
                              do_sample = True,
                              max_length = MAX_LEN,
                              temperature = 0.8,
                              top_k = 50,
                              top_p = 0.85
)

output = tokenizer.decode(sample_output[0], skip_special_tokens = True)

print(output)

Today is a bright day. But, to be fair, I am a little bit tired. I'm exhausted. But, I'm also happy to be back."

The 23-year-old's return was a big step forward for the Red Bulls. After playing just nine minutes in the season opener, he played the entire 90 minutes in the weekend's 2-0 win over Orlando City SC.

The Red Bulls were missing several key players in the backline and the midfield,


In [31]:
# feeding the generated output to the model again to the generate more

MAX_LEN = 50

input_sequence = "I believe, life is all about"

input_ids = tokenizer.encode(input_sequence, return_tensors='pt')

sample_output = GPT2.generate(
                              input_ids,
                              do_sample = True,
                              max_length = MAX_LEN,
                              temperature = 1.1,
                              #top_k = 50,
                              top_p = 0.7,
                              num_beams = 5,
                              no_repeat_ngram_size = 2,
)

output = tokenizer.decode(sample_output[0], skip_special_tokens = True)

print("First output:\n")
print(output)

input_ids2 = tokenizer.encode(output, return_tensors='pt')

sample_output2 = GPT2.generate(
                              input_ids2,
                              do_sample = True,
                              max_length = 2*MAX_LEN,
                              temperature = 0.8,
                              #top_k = 50,
                              top_p = 0.85,
                              num_beams = 5,
                              no_repeat_ngram_size = 2,
)

output2 = tokenizer.decode(sample_output2[0], skip_special_tokens = True)

print("\nNext output: \n")
print(output2)

First output:

I believe, life is all about the journey. The journey is the most important part of life, and that's what I'm going to do.

"I want to be the best player that I can be. I don't care what

Next output: 

I believe, life is all about the journey. The journey is the most important part of life, and that's what I'm going to do.

"I want to be the best player that I can be. I don't care what other people think about me, or what they think I should be doing. That's not my goal. My goal is to win a World Cup and I'll do whatever it takes to achieve that."
